# ATT simulation with (approximate) true value (genriesz)

This notebook demonstrates **ATT** estimation:

\[
\theta = \mathbb{E}[Y(1)-Y(0) \mid D=1].
\]

We generate a synthetic population with **heterogeneous treatment effects**, so
in general **ATT ≠ ATE**. We compute an approximate "true" ATT by Monte Carlo
from a large simulated population, and compare it to GRR-based estimators.

We assume the regressor has the form `X = [D, Z...]` where `D` is a binary
treatment indicator.


In [ ]:
import numpy as np

from genriesz import (
    grr_att,
    SquaredGenerator,
    PolynomialBasis,
    TreatmentInteractionBasis,
)

rng = np.random.default_rng(0)


## Data generating process

In [ ]:
def draw_population(n: int, d_z: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    Z = rng.normal(size=(n, d_z))

    logits = 0.7 * Z[:, 0] - 0.3 * Z[:, 1]
    e = 1.0 / (1.0 + np.exp(-logits))
    D = rng.binomial(1, e, size=n).astype(int)

    # Heterogeneous treatment effect
    tau = 1.0 + 0.5 * Z[:, 0]
    mu0 = 0.5 * Z[:, 0] + 0.25 * Z[:, 1] ** 2

    Y0 = mu0 + rng.normal(scale=1.0, size=n)
    Y1 = mu0 + tau + rng.normal(scale=1.0, size=n)
    Y = D * Y1 + (1 - D) * Y0

    X = np.column_stack([D.astype(float), Z])
    return X, Y, Y0, Y1, D, tau

# Large population for an approximate truth
X_pop, Y_pop, Y0_pop, Y1_pop, D_pop, tau_pop = draw_population(n=200_000, d_z=5, seed=1)
true_att = float(np.mean((Y1_pop - Y0_pop)[D_pop == 1]))
true_ate = float(np.mean(Y1_pop - Y0_pop))

print("Approx. true ATT (Monte Carlo):", true_att)
print("Approx. true ATE (Monte Carlo):", true_ate)


## Sample, then estimate ATT

In [ ]:
# Sample a dataset from the same DGP
X, Y, Y0, Y1, D, tau = draw_population(n=5000, d_z=5, seed=0)

# Basis on Z, then interact with D (works well for treatment-effect functionals)
psi = PolynomialBasis(degree=2, include_bias=True)
phi = TreatmentInteractionBasis(base_basis=psi)

gen = SquaredGenerator(C=0.0).as_generator()

res = grr_att(
    X=X,
    Y=Y,
    basis=phi,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res.summary_text())


## Compare estimates to the true ATT

The results below compare each estimator against the Monte Carlo "truth" from
the large simulated population.


In [ ]:
for key, est in res.estimates.items():
    err = est.estimate - true_att
    print(f"{key:>12s}: estimate={est.estimate: .6f},  error={err: .6f}")
